# Chapter 8.2 - Networks Using Blocks (VGG)

VGG turns CNN architecture into a block design discipline. Instead of treating every layer as a one-off decision, it repeatedly stacks small 3 by 3 convolutions followed by pooling. That makes the architecture easier to read, modify, and scale.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- define a VGG block and explain why it is reusable
- compare stacked 3 by 3 convolutions with a larger single convolution
- build a small VGG-style classifier from an architecture configuration
- trace spatial downsampling through repeated blocks
- debug the effect of forgetting padding in a VGG block


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current


## 8.2.0 The Problem This Notebook Solves

VGG's central lesson is not only a particular network. It is a way of designing networks:

```text
choose a simple block
repeat the block
increase channels across stages
reduce spatial resolution between stages
```

A block is a reusable module pattern. Chapter 6 introduced modules as software objects. VGG shows why this matters architecturally: repeated blocks let you express a deep model with a small, inspectable design vocabulary.


## 8.2.1 A VGG Block Is Repeated Local Processing Plus Downsampling

A typical VGG block contains:

- one or more 3 by 3 convolution layers with padding 1
- ReLU after each convolution
- 2 by 2 max pooling with stride 2

Padding 1 is important. A 3 by 3 convolution with padding 1 preserves height and width before pooling. Then pooling halves the spatial size. This makes the block's shape contract predictable.

Before running the cell, predict:

- Input shape: `(2, 3, 32, 32)`.
- Two padded convolutions should keep `32 by 32`.
- Pooling should produce `16 by 16`.
- Output channel count should be 8.


In [ ]:
def vgg_block(in_channels, out_channels, num_convs):
    layers = []
    current_channels = in_channels
    for _ in range(num_convs):
        layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        current_channels = out_channels
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)


block = vgg_block(3, 8, num_convs=2)
X = torch.randn(2, 3, 32, 32)
Y = block(X)

print("output shape:", shape(Y))
assert shape(Y) == (2, 8, 16, 16)


## 8.2.2 Two 3 by 3 Convolutions See a 5 by 5 Neighborhood

Stacking small kernels increases the effective receptive field. Receptive field means the region of the original input that can affect one output value.

Two stride-1 3 by 3 convolutions let a later output depend on a 5 by 5 region, but with an extra nonlinearity between the two convolutions. For equal input and output channel width, two 3 by 3 convolutions often use fewer parameters than one 5 by 5 convolution:

```text
two 3 by 3 layers: 2 * 3 * 3 * C * C
one 5 by 5 layer: 5 * 5 * C * C
```

This is a design tradeoff: VGG prefers a regular stack of small local operations.


In [ ]:
channels = 16
two_3x3 = 2 * 3 * 3 * channels * channels
one_5x5 = 5 * 5 * channels * channels

print("two 3x3 weights:", two_3x3)
print("one 5x5 weights:", one_5x5)
print("saving:", one_5x5 - two_3x3)

assert two_3x3 < one_5x5


## 8.2.3 Build VGG From an Architecture Configuration

An architecture configuration is a compact description of repeated stages. In this notebook, each tuple means:

```text
(number of convolutions in the block, output channels)
```

The builder below converts that list into an executable `nn.Sequential`. This is the bridge from design vocabulary to framework code.

Before running the cell, predict:

- Three blocks with pooling should reduce 64 to 32 to 16 to 8.
- The final adaptive average pool should make the dense head independent of the exact final spatial size.
- The logits should have shape `(2, 10)`.


In [ ]:
def make_vgg(in_channels, arch, num_classes=10):
    layers = []
    current_channels = in_channels
    for num_convs, out_channels in arch:
        layers.append(vgg_block(current_channels, out_channels, num_convs))
        current_channels = out_channels
    layers += [
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(current_channels, num_classes),
    ]
    return nn.Sequential(*layers)


tiny_vgg = make_vgg(1, [(1, 8), (1, 16), (2, 32)])
X = torch.randn(2, 1, 64, 64)
rows, logits = trace_module_shapes(tiny_vgg, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)


## 8.2.4 Blocks Make Parameter Accounting Local

When a network is built from blocks, you can inspect each block separately. This matters because modern CNNs are not just long lists of layers; they are systems of repeated components.

The cell counts trainable scalars per top-level stage. This is not the same as measuring runtime speed, but it reveals where model capacity lives.


In [ ]:
for name, layer in tiny_vgg.named_children():
    print(name, layer.__class__.__name__, count_parameters(layer))

total = count_parameters(tiny_vgg)
print("total trainable scalars:", total)

assert total == sum(count_parameters(layer) for layer in tiny_vgg.children())


## 8.2.5 Break It Deliberately: Forget Padding

The VGG block pattern depends on padded 3 by 3 convolutions preserving spatial size before pooling. If you forget padding, every convolution shrinks height and width before the pooling layer runs.

The forward pass may still run, which makes this bug subtle. The architecture no longer has the shape story you intended.


In [ ]:
bad_block = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3), nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

good_Y = block(torch.randn(2, 3, 32, 32))
bad_Y = bad_block(torch.randn(2, 3, 32, 32))

print("good block:", shape(good_Y))
print("bad block:", shape(bad_Y))

try:
    assert shape(bad_Y)[-2:] == (16, 16)
except AssertionError:
    print("The block ran, but it violated the intended VGG shape contract.")


## 8.2 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What is a VGG block?
2. Why can repeated 3 by 3 convolutions be preferable to one larger convolution?
3. Why does padding matter inside a VGG block?
4. What does an architecture configuration buy you as a programmer?
5. Why is a running forward pass not enough to prove the architecture is correct?
